In [12]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, case, distinct, func, literal, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "TEST"  # Lavoriamo in test per essere veloci
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

In [13]:
SECTOR_NAME = "Quantum Computing"
SECTOR_CPC_PREFIXES = (
    "G06N10/00",  
    "G06N10/20",  
    "G06N10/40",  
    "G06N10/60",  
    "B82Y 10/00", 
    "H10N 60/",   
)

SECTOR_KEYWORDS = (
    "quantum computer",
    "quantum computing",
    "qubit",
    "quantum processor",
    "quantum algorithm",
    "superconducting qubit",
    "quantum error correction",
)

def symbol_prefix_condition(symbol_column, codes):
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))

def keyword_condition(terms):
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))

def sector_family_ids():
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
    by_cpc = (
        db.query(family_id)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES))
    )
    if not SECTOR_KEYWORDS:
        return by_cpc.distinct()
    by_keywords = (
        db.query(family_id)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(SECTOR_KEYWORDS))
    )
    return by_cpc.union(by_keywords)

sector_ids = sector_family_ids()

In [14]:
# 1. Troviamo prima solo gli ID di famiglia campionati (velocissimo)
fam_pos = db.query(TLS201_APPLN.docdb_family_id).filter(TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select())).distinct().limit(50)
df_pos = patstat.df(fam_pos)
df_pos["query_label"] = 1

fam_neg = db.query(TLS201_APPLN.docdb_family_id).filter(~TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select())).distinct().limit(50)
df_neg = patstat.df(fam_neg)
df_neg["query_label"] = 0

# Uniamo e prendiamo un campione casuale di 100 famiglie
df_validation = pd.concat([df_pos, df_neg]).sample(n=100, random_state=42).reset_index(drop=True)

# 2. Adesso, recuperiamo titolo e abstract SOLO per queste 100 famiglie scelte (operazione istantanea)
family_ids_list = df_validation["docdb_family_id"].tolist()

text_query = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract
    )
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(family_ids_list))
)
df_texts = patstat.df(text_query).drop_duplicates(subset=["docdb_family_id"])

# Uniamo i testi al nostro campione di 100
df_validation = df_validation.merge(df_texts, on="docdb_family_id", how="left")

# Aggiungiamo la colonna vuota per la validazione manuale
df_validation["manual_label"] = ""

# Salviamo il CSV
validation_csv_path = OUTPUT_DIR / "validation_sample_100.csv"
df_validation.to_csv(validation_csv_path, index=False)

print(f"Validation file: {validation_csv_path}")


Fatto! File CSV generato in un attimo: outputs/step0/validation_sample_100.csv


,docdb_family_id,query_label,appln_title,appln_abstract,manual_label
0,3446469,0,None,None,
1,161392,0,None,None,
2,3407768,0,None,None,
3,80034724,1,Systems and methods for post-quantum cryptogra...,"Systems, apparatuses, methods, and computer pr...",
4,75438227,1,Dual pole high temperature superconductive par...,A dual pole high temperature superconductive p...,
